# Day 13 Project Solution: Document Q&A with RAG

In [ ]:
import chromadb
import ollama

MODEL = "llama3.2"
EMBED_MODEL = "nomic-embed-text"
CHUNK_SIZE = 300
OVERLAP = 40
TOP_K = 3

GROUNDING_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions strictly from the "
    "provided context. If the answer is not in the context, respond with: "
    "'I don't know based on the provided documents.' "
    "Do not use any knowledge outside the context."
)

In [ ]:
def chunk_document(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be less than chunk_size")
    chunks, step, start = [], chunk_size - overlap, 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += step
    return chunks


def build_rag_prompt(question: str, chunks: list[str]) -> str:
    context = "\n\n---\n\n".join(chunks)
    return (
        f"Use the following context to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )

In [ ]:
def index_corpus(docs: list[dict], collection, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> int:
    total = 0
    for doc in docs:
        chunks = chunk_document(doc["text"], chunk_size, overlap)
        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc['source']}_{i:04d}"
            emb = ollama.embeddings(model=EMBED_MODEL, prompt=chunk)
            collection.add(
                ids=[chunk_id],
                embeddings=[emb["embedding"]],
                documents=[chunk],
                metadatas=[{"source": doc["source"], "chunk_index": i}],
            )
            total += 1
    return total


def retrieve_context(question: str, collection, top_k: int = TOP_K) -> list[str]:
    q_emb = ollama.embeddings(model=EMBED_MODEL, prompt=question)
    results = collection.query(
        query_embeddings=[q_emb["embedding"]],
        n_results=top_k,
    )
    return results["documents"][0]


def rag_answer(question: str, collection, model: str = MODEL) -> str:
    chunks = retrieve_context(question, collection, top_k=TOP_K)
    user_msg = build_rag_prompt(question, chunks)
    messages = [
        {"role": "system", "content": GROUNDING_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]

In [ ]:
CORPUS = [
    {
        "source": "python_history.txt",
        "text": (
            "Python is a high-level, general-purpose programming language. "
            "Its design philosophy emphasises code readability, notably through the use of significant indentation. "
            "Python is dynamically typed and garbage-collected. "
            "It supports multiple programming paradigms, including structured, object-oriented and functional programming. "
            "Python was created by Guido van Rossum, who began working on it in the late 1980s as a successor to the ABC programming language. "
            "Python 2.0 was released in 2000. Python 3.0 was released in 2008. "
            "Python consistently ranks as one of the most popular programming languages. "
        ) * 2,
    },
    {
        "source": "machine_learning.txt",
        "text": (
            "Machine learning is a subfield of artificial intelligence that focuses on building systems that learn from data. "
            "Supervised learning trains a model on labelled examples — each input has a known correct output. "
            "Unsupervised learning finds patterns in data without labels. "
            "Reinforcement learning trains agents through rewards and penalties. "
            "Neural networks are a family of machine learning models loosely inspired by the brain. "
            "Deep learning uses neural networks with many layers to learn complex representations. "
            "Common supervised learning algorithms include linear regression, logistic regression, and decision trees. "
        ) * 2,
    },
    {
        "source": "vector_databases.txt",
        "text": (
            "A vector database stores data as high-dimensional vectors (embeddings) and enables fast similarity search. "
            "Unlike traditional databases that match on exact values, vector databases retrieve by semantic closeness. "
            "ChromaDB is an open-source vector database that runs locally with no external dependencies. "
            "Approximate nearest-neighbour search (ANN) makes queries fast even over millions of vectors. "
            "Vector databases are a core component of Retrieval-Augmented Generation (RAG) pipelines. "
            "Metadata filtering lets you narrow a vector search to a subset of documents before scoring by similarity. "
        ) * 2,
    },
]

client = chromadb.Client()
collection = client.get_or_create_collection("day13_rag")
chunk_count = index_corpus(CORPUS, collection)
print(f"Indexed {chunk_count} chunks from {len(CORPUS)} documents.")

In [ ]:
QUESTIONS = [
    "Who created Python?",
    "What is supervised learning?",
    "What is ChromaDB used for?",
    "What is the capital of France?",  # out-of-context
]

print("=" * 60)
print("Document Q&A — RAG Pipeline")
print("=" * 60)
for question in QUESTIONS:
    print(f"\nQ: {question}")
    answer = rag_answer(question, collection)
    print(f"A: {answer}")
    print("-" * 40)

In [ ]:
print("\n✅ Day 13 project complete.")
print(f"   Corpus: {len(CORPUS)} documents → {chunk_count} chunks indexed")
print(f"   Questions answered: {len(QUESTIONS)}")
print("   RAG pipeline: chunk_document → index_corpus → retrieve_context → build_rag_prompt → rag_answer")
print("   Deliverable: questions answered from local documents with grounded LLM responses.")